In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import os

fifa_ranking_2022 = pd.read_csv('fifa_ranking_2022-10-06.csv')

fifa_ranking_2026 = pd.read_csv('fifa_ranking_2026-06-08.csv')

matches = pd.read_csv('matches_1930_2022.csv')

schedule_2026 = pd.read_csv('schedule_2026.csv')

world_cup = pd.read_csv('world_cup.csv')

print("Successfully loaded all datasets.")


In [ ]:
# Display the first few rows of each dataset
print("FIFA Ranking 2022:")
print(fifa_ranking_2022.head().to_string(index=False))

print("\nFIFA Ranking 2026:")
print(fifa_ranking_2026.head().to_string(index=False))

print("\nMatches 1930-2022:")
print(matches.head().to_string(index=False))

print("\nMatches 2026:")
print(schedule_2026.head().to_string(index=False))

print("\nWorld Cup History:")
print(world_cup.head().to_string(index=False))

In [ ]:
schedule_2026_missing_val = schedule_2026.isnull().sum()
fifa_ranking_2022_missing_val = fifa_ranking_2022.isnull().sum()
fifa_ranking_2026_missing_val = fifa_ranking_2026.isnull().sum()
matches_missing_val = matches.isnull().sum()
world_cup_missing_val = matches.isnull().sum()
missing_total = schedule_2026_missing_val + fifa_ranking_2022_missing_val + fifa_ranking_2026_missing_val + matches_missing_val + world_cup_missing_val

print("Display missing values in all data files.")
print(missing_total)

# Filling missing values with 0
schedule_2026 = schedule_2026.fillna(0)
fifa_ranking_2022 = fifa_ranking_2022.fillna(0)
fifa_ranking_2026 = fifa_ranking_2026.fillna(0)
matches = matches.fillna(0)
world_cup = world_cup.fillna(0)


team_mapping = {
    "United States" : "USA",
    "Bosnia-Herzegovina" : "Bosnia and Herzegovina",
    "Congo DR" : "Congo"
}

schedule_2026['home_team'] = schedule_2026["home_team"].replace(team_mapping)
schedule_2026['away_team'] = schedule_2026["away_team"].replace(team_mapping)



In [ ]:
merge_ranking = pd.merge(fifa_ranking_2026, fifa_ranking_2022, on='team', suffixes=('_2026', '_2022'))

merge_ranking['rank_change'] = merge_ranking['rank_2022'] - merge_ranking['rank_2026']

# Display the merged ranking
print(merge_ranking[['team', 'rank_2022', 'rank_2026', 'rank_change']].head().to_string(index=False))

In [ ]:
# World Cup summary
print('World Cup Data Head.')
print(world_cup.head().to_string(index=False))

# Display Championship
champion_table = world_cup.groupby('Champion')['Year'].apply(lambda x: sorted(x))
champion_table = champion_table.reset_index()
champion_table['Count'] = champion_table['Year'].apply(len)
champion_table = champion_table.sort_values('Count', ascending=False)[['Champion', 'Count', 'Year']]
champion_table = champion_table.reset_index(drop=True)

print("\nWorld Cup Champions Frequency")
print(champion_table)

champion_counts = world_cup['Champion'].value_counts()
plt.figure(figsize=(12, 6))
sns.barplot(x=champion_counts.index, y=champion_counts.values, hue=champion_counts.index, palette='viridis', legend=False)
plt.title('World Cup Champions Frequency')
plt.xticks(rotation=45)
plt.xlabel('Champion')
plt.ylabel('Cups')
plt.show()

In [ ]:
matches['Date'] = pd.to_datetime(matches['Date'], errors='coerce')
print("Display match dataset Date column info, after modifying.")
print(matches['Date'].describe())

# Display mathces per year
matches_per_year = matches.groupby('Year').size()
plt.figure(figsize=(12,6))
sns.lineplot(x=matches_per_year.index, y=matches_per_year.values, marker='o')
plt.title('Numer of matches per year')
plt.xlabel('Yead')
plt.ylabel('Number of matches')
plt.show()

In [ ]:
schedule_2026['Date'] = pd.to_datetime(schedule_2026['Date'], errors='coerce')
print("Date schedules 2026 after modifying.")
print(schedule_2026['Date'].describe())

print("\nSchedule 2026 head.")
print(schedule_2026.head().to_string(index=False))

plt.figure(figsize=(12,6))
sns.countplot(data=schedule_2026, x='Date', palette='Set2', order=sorted(schedule_2026['Date'].unique()), hue='Date', legend=False)
plt.title("Number of Matches per Day - Tournament Schedule 2026")
plt.xlabel('Date')
plt.ylabel('Number of matches')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
home_rankings = fifa_ranking_2026[[
    "team",
    "rank",
    "points"
]].rename(columns={
    "team": "home_team",
    "rank": "home_rank",
    "points": "home_points"
})

matches_2026 = schedule_2026.merge(
    home_rankings,
    on="home_team",
    how="left"
)

away_rankings = fifa_ranking_2026[[
    "team",
    "rank",
    "points"
]].rename(columns={
    "team": "away_team",
    "rank": "away_rank",
    "points": "away_points"
})

matches_2026 = matches_2026.merge(
    away_rankings,
    on="away_team",
    how="left"
)

matches_2026["point_diff"] = matches_2026["home_points"] - matches_2026["away_points"]
matches_2026["rank_diff"] = matches_2026["home_rank"] - matches_2026["away_rank"]

print(matches_2026[["point_diff", "rank_diff"]].head().to_string(index=False))
print(matches_2026.head().to_string(index=False))